# Quantum simulation (Trotter) — trimero a catena aperta

Mirror diretto, per la topologia a catena, di `quantum_simulation_trimero_anello_trotter.ipynb`.
Stessa struttura a tre livelli (livello 0 esatto, livello 1 = splitting di $H_{ex}$,
livello 2 = splitting di $H_{DM}$), con due differenze qualitative rispetto all'anello:

1. **Un solo grado di liberta' DM** ($D_{12}=-D_{23}=D$, fissato da $P_{13}$), non due opzioni A/B.
2. **L'ordine dei due bond puo' essere alternato** fra passi consecutivi: il termine
   chirale di livello 1 cambia segno con l'ordine, quindi alternare lo cancella al
   leading order a parita' esatta di gate — non c'e' un fenomeno equivalente
   nell'anello (tre bond, non due).

Punto di lavoro: **provvisorio**, trovato per scansione (stesso status di $R_0$
per l'anello), non da una derivazione fisica principiata.

In [ ]:
import numpy as np
import scipy.linalg as sla
from qiskit.quantum_info import Statevector, Operator

from trotter_trimero_catena import (
    trotter_circuit, H_parts, U_exact, sz_tot_matrix,
    chi_operator, theta_operator, bond_exchange_matrix, bond_dm_matrix,
    PSI0,
)
from trimer_chain_exact import critical_field, dm_min_gap

SZ = sz_tot_matrix()
np.set_printoptions(precision=4, suppress=True)

## 1. Self-test del modulo

Eseguito qui per intero prima di qualunque analisi fisica.

In [ ]:
import trotter_trimero_catena as ttc
ttc._self_test()

## 2. Ricerca del punto di lavoro

Stesso criterio usato per l'anello: cercare un punto con **piu' modi spettrali di
ampiezza comparabile** nella decomposizione di Bohr di $\langle S_z^{tot}\rangle(t)$,
non solo un singolo modo dominante — altrimenti la dinamica e' un semplice coseno,
poco interessante per lo studio di Trotter.

Regione naturale: attorno al campo critico $b_c=3J$ (crossing esatto senza DM,
teorema di Kramers), dove il DM apre un vero anti-crossing (vedi
`analisi_dm_trimero_catena.tex`).

In [ ]:
def mode_amplitudes(J, b, D):
    """Decomposizione di Bohr: ampiezza a_kl = 2|c_k* c_l (Sz_tot)_kl| e gap_kl per ogni coppia."""
    H0, HDM = H_parts(J, b, D)
    w, v = np.linalg.eigh(H0 + HDM)
    modes = []
    for k in range(8):
        for l in range(k + 1, 8):
            a = 2 * abs(v[:, k].conj() @ SZ @ v[:, l])
            if a > 1e-8:
                modes.append((a, w[l] - w[k], k, l))
    modes.sort(reverse=True)
    return modes

J = 1.0
bc = critical_field(J)
print(f"b_c = {bc}")
print(f"{'b':>7} {'D':>6} {'a1':>9} {'a2':>9} {'a2/a1':>7}")
for D in [0.05, 0.1, 0.2, 0.3, 0.5]:
    for b in [bc - 0.05, bc - 0.02, bc, bc + 0.02, bc + 0.05]:
        modes = mode_amplitudes(J, b, D)
        a1, a2 = modes[0][0], modes[1][0]
        print(f"{b:7.3f} {D:6.3f} {a1:9.4f} {a2:9.4f} {a2/a1:7.3f}")

**Osservazione**: a $b=b_c$ esatto, $a_2/a_1 \to 1$ per ogni $D$ testato — i due modi
dominanti sono quasi degeneri in ampiezza li'. Scelgo come punto di lavoro il minimo
**vero** del gap (non $b_c$ fisso: stesso errore metodologico gia' documentato per
l'anello e per la catena in `analisi_dm_trimero_catena.tex`), con $D=0.3$: gap
sufficientemente grande da essere risolto in tempo ragionevole, non cosi' grande da
rendere banale la dinamica.

In [ ]:
D_lavoro = 0.3
gap_min, b_lavoro = dm_min_gap(J, D_lavoro)
print(f"Punto di lavoro S0: J={J}, b={b_lavoro:.7f}, D={D_lavoro}")
print(f"gap all'anti-crossing = {gap_min:.6f}")

modes = mode_amplitudes(J, b_lavoro, D_lavoro)
print("\nModi spettrali (ampiezza, gap):")
for a, g, k, l in modes:
    print(f"  a={a:.5f}  gap={g:.5f}   ({k},{l})")

Due modi dominanti quasi degeneri in ampiezza ($a\approx2.0$, gap $0.847$ e
$1.469$) — la stessa $\langle S_z^{tot}\rangle(t)$ mostrera' un **battimento** di
periodo $2\pi/|{\rm gap}_2-{\rm gap}_1| \approx 10.1$, non un singolo coseno.
Punto **provvisorio** — nessuna conferma del relatore, analogo status di $R_0$
per l'anello.

## 3. Traiettoria esatta di $\langle S_z^{tot}\rangle(t)$

In [ ]:
def sz_traj_exact(J, b, D, t_max, n_t=300, psi0=None):
    H0, HDM = H_parts(J, b, D)
    H = H0 + HDM
    if psi0 is None:
        psi0 = Statevector.from_label('010').data
    ts = np.linspace(0, t_max, n_t)
    vals = np.array([np.real(
        (sla.expm(-1j * H * t) @ psi0).conj() @ SZ @ (sla.expm(-1j * H * t) @ psi0)
    ) for t in ts])
    return ts, vals

ts, sz = sz_traj_exact(J, b_lavoro, D_lavoro, 30.0)
print(f"escursione picco-picco su |010>: {sz.max()-sz.min():.4f}")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(7,3))
ax.plot(ts, sz)
ax.set_xlabel('t'); ax.set_ylabel(r'$\langle S_z^{tot}\rangle(t)$')
ax.set_title(f'Traiettoria esatta, S0: J={J}, b={b_lavoro:.4f}, D={D_lavoro}')
fig.tight_layout()
plt.show()

## 4. Convergenza Trotter: fisso vs alternato

Nota metodologica prima di procedere: **|000> e' cieco all'errore di livello 1**
(autostato di ogni bond, vedi self-test 6/7 del modulo) — un test di convergenza
su questo stato misurerebbe solo il livello 2. Uso quindi `|010>` come stato di
prova, coerente col resto del modulo.

In [ ]:
psi0 = Statevector.from_label('010').data
t_ref = 20.0
ref = U_exact(J, b_lavoro, D_lavoro, t_ref) @ psi0

Ns = [500, 1000, 2000, 4000, 8000]
rows = []
prev = [None, None]
print(f"{'N':>5} {'infed. fisso':>14} {'':>6} {'infed. alternato':>16} {'':>6} {'guadagno':>9}")
for N in Ns:
    out = []
    for alt in (False, True):
        U = Operator(trotter_circuit(J, b_lavoro, D_lavoro, t_ref, N, alternate=alt)).data
        psi = U @ psi0
        out.append(1 - abs(np.vdot(ref, psi)) ** 2)
    rf = f"x{prev[0]/out[0]:.2f}" if prev[0] else "  -"
    ra = f"x{prev[1]/out[1]:.2f}" if prev[1] else "  -"
    print(f"{N:5d} {out[0]:14.4e} {rf:>6} {out[1]:16.4e} {ra:>6} {out[0]/out[1]:9.2f}")
    rows.append((N, *out))
    prev = out

**Lettura onesta**: il guadagno dell'alternanza cresce (da $\times1.3$ a
$N=500$ fino a $\times25$ a $N=8000$) ma **non** raggiunge il rapporto pulito
$\times16$ per raddoppio di $N$ osservato a $D=0$ nel modulo — qui il livello 2
e i termini incrociati $H_{ex}/H_{DM}$, $S_z^{tot}/H_{DM}$ restano $O(1/N^2)$ e
limitano il guadagno asintotico. Coerente con quanto gia' registrato in
`log_decisioni.md`: l'alternanza cancella *solo* il termine chirale di livello 1,
non l'intero errore Trotter.

## 5. Circuito: un passo, punto di lavoro S0

In [ ]:
qc = trotter_circuit(J, b_lavoro, D_lavoro, 1.0, 1)
print(f"gate totali per un passo: {qc.size()}")
qc.draw(output="mpl", fold=-1, style={"name": "iqp"})

## 6. Conclusione

Punto di lavoro S0 ($J{=}1$, $b{=}3.0073414$, $D{=}0.3$) trovato per scansione
attorno al campo critico $b_c=3J$, con due modi spettrali quasi degeneri in
ampiezza — analogo a $R_0$ per l'anello, con lo stesso status **provvisorio**.

Risultato originale di questa fase: l'alternanza dell'ordine dei bond, possibile
solo nella catena (due bond, non tre), da' un guadagno reale ma **parziale** in
presenza di DM — netto ($O(1/N^4)$) solo a $D=0$, dove isola il solo livello 1.

Prossimo passo: Fase 5 (VQE+DM) per la catena e correlatori dinamici, secondo
`domande_relatore.md`.